In [ ]:
#imports
from sklearn.datasets import fetch_openml
from sklearn.metrics import confusion_matrix
import numpy as np
from brian2 import *
from tqdm.notebook import tqdm
import os
import gc

prefs.codegen.target = 'cython'
# prefs.codegen.target = 'numpy'

from network_garg import *
from utils import *
from eqn_garg import *

In [ ]:
mnist = fetch_openml('mnist_784', data_home='../mnist_data/', as_frame=False, parser='liac-arff')

X = mnist.data
y = mnist.target
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
weights_path = '../trained_model/garg/weights.npy'
thetas_path = '../trained_model/garg/thetas.npy'
delays_path = '../trained_model/garg/delays.npy'
labels_path = '../trained_model/garg/neuron_labels.npy'

In [ ]:
net, inp_group, spike_monitor = build_network_test()
net['s_inp_exc'].w = np.load(weights_path)
net['exc'].theta = np.load(thetas_path) * volt
net['s_inp_exc'].delay = np.load(delays_path) * second
net.store('inference_init') # takes a snapshot of the network

In [ ]:
spike_counts = np.zeros((n_e, 10))
num_examples = 10000

for i in tqdm(range(num_examples)):
    
    net.restore('inference_init') # Reset network
    image = X_train[i]
    current_label = int(y_train[i])
    
    pixel_values = np.clip(image / 255.0, 0.0, 1.0)
    inp_group.I = pixel_values * volt
    
    net.run(350 * ms)
    
    spike_counts[:, current_label] += spike_monitor.count

neuron_labels = np.argmax(spike_counts, axis=1)
dead_neurons = np.sum(spike_counts, axis=1) == 0
neuron_labels[dead_neurons] = -1  # assign -1 to inactive neurons

np.save(labels_path, neuron_labels)

In [ ]:
num_test_images = 10000
predictions = []
actuals = []

for i in tqdm(range(num_test_images)):
    
    net.restore('inference_init')
    
    image = X_test[i]
    true_label = int(y_test[i])
    actuals.append(true_label)
    
    pixel_values = np.clip(image / 255.0, 0.0, 1.0)
    inp_group.I = pixel_values * volt
    net.run(350 * ms)
    
    class_spikes = np.zeros(10)
    for digit in range(10):
        neurons_in_class = (neuron_labels == digit)
        class_spikes[digit] = np.sum(spike_monitor.count[neurons_in_class])
    
    if np.sum(class_spikes) > 0:  
        predicted_label = np.argmax(class_spikes)
    else:
        predicted_label = -1  # Network stayed silent
        
    predictions.append(predicted_label)

In [ ]:
predictions = np.array(predictions)
actuals = np.array(actuals)

valid_idx = predictions != -1
correct_predictions = np.sum(predictions[valid_idx] == actuals[valid_idx])
accuracy = (correct_predictions / num_test_images) * 100.0

print(f"\nFinal Accuracy: {accuracy:.2f}% ({correct_predictions} / {num_test_images})")
print(f"Silent Images: {np.sum(predictions == -1)}")

In [ ]:
conf_matrix = confusion_matrix(actuals[valid_idx], predictions[valid_idx], labels=np.arange(10))
plot_confusion_matrix(conf_matrix)